# Multi-Hop Retrieval — Colab Training

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free tier) or better
2. Upload MuSiQue JSONL files to Google Drive → `My Drive/musique_data/`
   - `musique_ans_v1.0_train.jsonl`  (~230 MB)
   - `musique_ans_v1.0_dev.jsonl`    (~30 MB)
3. Fill in `REPO_URL` in the cell below with your GitHub repo URL

**What this notebook does:**
- Trains **Model 1**: `ComplementEncoder` — joint BERT, chain contrastive loss, query-agnostic
- Trains **Model 2**: `ColBERTScorer` — MaxSim query scoring, Model 1 frozen
- Saves both checkpoints to `My Drive/musique_data/` for later use

In [ ]:
# ── [EDIT THIS] ────────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"  # your repo
REPO_NAME = "multihop-retrieval"   # folder name after git clone
DRIVE_DATA_DIR = "/content/drive/MyDrive/musique_data"  # where you uploaded JSONL
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Go to Runtime → Change runtime type → T4 GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")
print("CUDA OK — ready to train")

In [ ]:
# 2. Clone repo and install dependencies
import os

if not os.path.isdir(f"/content/{REPO_NAME}"):
    os.system(f"git clone {REPO_URL} /content/{REPO_NAME}")
else:
    print("Repo already cloned — pulling latest")
    os.system(f"cd /content/{REPO_NAME} && git pull")

os.chdir(f"/content/{REPO_NAME}/retrieval")
print("Working directory:", os.getcwd())

In [ ]:
# 3. Install Python dependencies
!pip install -q -r requirements.txt
print("Dependencies installed")

In [ ]:
# 4. Mount Google Drive and verify MuSiQue data
from google.colab import drive
drive.mount('/content/drive')

import os
train_file = f"{DRIVE_DATA_DIR}/musique_ans_v1.0_train.jsonl"
dev_file   = f"{DRIVE_DATA_DIR}/musique_ans_v1.0_dev.jsonl"

assert os.path.exists(train_file), f"NOT FOUND: {train_file}\nUpload musique_ans_v1.0_train.jsonl to My Drive/musique_data/"
assert os.path.exists(dev_file),   f"NOT FOUND: {dev_file}\nUpload musique_ans_v1.0_dev.jsonl to My Drive/musique_data/"

print(f"train : {os.path.getsize(train_file)/1e6:.1f} MB  OK")
print(f"dev   : {os.path.getsize(dev_file)/1e6:.1f} MB  OK")

In [ ]:
# 5. Symlink Drive data into the expected local path
import os

local_data = "/content/" + REPO_NAME + "/retrieval/data/musique"
os.makedirs(local_data, exist_ok=True)
os.makedirs("/content/" + REPO_NAME + "/retrieval/models", exist_ok=True)
os.makedirs("/content/" + REPO_NAME + "/retrieval/cache", exist_ok=True)

for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{DRIVE_DATA_DIR}/{fname}"
    dst = f"{local_data}/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: {'symlinked' if os.path.islink(dst) else 'exists'}")

print("Data paths ready")

In [ ]:
# 6. Smoke test — verify data loader and build functions work (200 examples)
!python data_loader.py
print("\nData loader smoke test passed")

---
## Train Model 1 — ComplementEncoder

- Joint BERT encoding of `[CLS] A [SEP] B [SEP]`
- Extracts B-side token matrix (B conditioned on A)
- Chain contrastive loss: B's complement should point toward next-hop C
- Trains on 3/4-hop MuSiQue only (~7K questions)
- Expected time: **~90 min** on T4

In [ ]:
# 7. Train Model 1 (full training)
!python model1_train.py
print("\nModel 1 training complete")

In [ ]:
# 8. Save Model 1 checkpoint to Drive
import shutil, os

src = f"/content/{REPO_NAME}/retrieval/models/model1_complement.pt"
dst = f"{DRIVE_DATA_DIR}/model1_complement.pt"

assert os.path.exists(src), f"Checkpoint not found at {src} — did training complete?"
shutil.copy(src, dst)
print(f"Model 1 saved to Drive: {dst}")
print(f"  Size: {os.path.getsize(dst)/1e6:.1f} MB")

---
## Train Model 2 — ColBERTScorer

- Query encoder initialised from `colbert-ir/colbertv2.0` (MS MARCO)
- MaxSim scoring: each Q token finds best match in B's complement tokens
- Model 1 is **frozen** — preserves query-agnostic property
- Trains on all hop counts (2/3/4-hop MuSiQue)
- Expected time: **~60 min** on T4

In [ ]:
# 9. Train Model 2 (full training)
!python model2_train.py
print("\nModel 2 training complete")

In [ ]:
# 10. Save Model 2 checkpoint to Drive
import shutil, os

src = f"/content/{REPO_NAME}/retrieval/models/model2_scorer_final.pt"
dst = f"{DRIVE_DATA_DIR}/model2_scorer_final.pt"

assert os.path.exists(src), f"Checkpoint not found at {src} — did training complete?"
shutil.copy(src, dst)
print(f"Model 2 saved to Drive: {dst}")
print(f"  Size: {os.path.getsize(dst)/1e6:.1f} MB")

In [ ]:
# 11. List everything saved to Drive
import os
print("Files in Drive data dir:")
for f in sorted(os.listdir(DRIVE_DATA_DIR)):
    size_mb = os.path.getsize(f"{DRIVE_DATA_DIR}/{f}") / 1e6
    print(f"  {f:45s}  {size_mb:7.1f} MB")

---
## Done

Checkpoints are saved to `My Drive/musique_data/`:
- `model1_complement.pt` — ComplementEncoder weights
- `model2_scorer_final.pt` — ColBERTQueryEncoder weights

To run inference locally:
1. Download both `.pt` files from Drive to `retrieval/models/`
2. Run `python run_full_system.py`